# はじめに

このノートでは、PyTorch Geometricを利用してグラフニューラルネットワークを構築することを目指す。最初はPyTorchの基礎からはじめ、グラフニューラルネットワーク、PyTorch Geometricと内容を進めていく。

前回は、PyTorch Geometricを使って競艇の「展開」を捉えるための特徴量をお試しで作っていたが、今回はLightGBMが特徴量として利用するための埋め込みを計算する。これはグラフニューラルネットワークのベースコードなので、そのまま使っているわけではない。モデルの改良を行う際に、処理の内容を思い出せるように内容をまとめておくことが目的。グラフニューラルネットワークは学習中なので、やっている内容が頓珍漢な可能性は否定できない。

下記はグラフニューラルネットワークを理解するための参考サイト。

- [グラフニューラルネットワーク | 佐藤 竜馬](https://www.amazon.co.jp/%E3%82%B0%E3%83%A9%E3%83%95%E3%83%8B%E3%83%A5%E3%83%BC%E3%83%A9%E3%83%AB%E3%83%8D%E3%83%83%E3%83%88%E3%83%AF%E3%83%BC%E3%82%AF-%E6%A9%9F%E6%A2%B0%E5%AD%A6%E7%BF%92%E3%83%97%E3%83%AD%E3%83%95%E3%82%A7%E3%83%83%E3%82%B7%E3%83%A7%E3%83%8A%E3%83%AB%E3%82%B7%E3%83%AA%E3%83%BC%E3%82%BA-%E4%BD%90%E8%97%A4-%E7%AB%9C%E9%A6%AC/dp/4065347823)
- [グラフ深層学習のすゝめ。 - YouTube](https://www.youtube.com/watch?v=7rgXi3Xp6NI)
- [GCN — グラフ道場](https://yuya-s.github.io/GraphDojo/01GCN.html)
- [Tutorial 6: Basics of Graph Neural Networks](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/06-graph-neural-networks.html)
- [Static and Dynamic Attention: Implications for Graph Neural Networks](https://medium.com/data-science/static-and-dynamic-attention-implications-for-graph-neural-networks-eda0d9d7b60a)
- [CS224W | Home](https://web.stanford.edu/class/cs224w/index.html)
- [nn.labml.ai/ja/graphs](https://nn.labml.ai/#:~:text=%E2%9C%A8%20Graph%20Neural%20Networks)
- [Understanding Convolutions on Graphs](https://distill.pub/2021/understanding-gnns/)
- [A Gentle Introduction to Graph Neural Networks](https://distill.pub/2021/gnn-intro/)

## 事前の準備

計算に必要なライブラリを読み込んでおく。

In [60]:
import argparse
import datetime
import math
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import to_dense_adj

パスの指定とパラメタやモデルの学習や予測で固定の値は`Config`にまとめておく。

In [61]:
# SCRIPT_DIR = Path(__file__).parent
# logger = setup_logger(__name__, log_file=SCRIPT_DIR / ".." / "logs" / "embeding.log")
DATA_DIR = Path('~/Documents/statistical_note/note_PyTorch08').expanduser().resolve() 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass
class Config:
    train_csv_path: Path = DATA_DIR / "02_features" / "features.csv"
    pred_csv_path: Path = DATA_DIR / "02_features" / "features_prediction.csv"
    model_path: Path = DATA_DIR / "03_train" / "tenkai_gat_model.pt"
    train_emb_out_path: Path = (DATA_DIR / "02_features" / "tenkai_gat_embedding_train.csv") # fmt: skip
    pred_emb_out_path: Path = (DATA_DIR / "02_features" / "tenkai_gat_embedding_prediction.csv") # fmt: skip
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 20
    seed: int = 42
    num_boats: int = 6
    feature_cols: list[str] = field(
        default_factory=lambda: [
            "StartTime_3races","StartTime_5races","StartTime_10races",
            "glicko2_rating","glicko2_rating_waku",
            "Course1Win1Rate","Course1SasareRate","Course1MakurareRate","Course1MakuriWinRate","Course2MakuriWinRate","Course3MakuriWinRate","Course1SashiWinRate","Course2SashiWinRate","Course3SashiWinRate","Course2NigashiRate", # fmt:skip
            "glicko2_rating_relative","glicko2_rating_waku_relative","TeibanStartTimeMean_relative","ZenkokuWinRate_relative","LocalWinRate_relative","Motor2WinRate_relative","Boat2WinRate_relative", # fmt:skip
        ]
    )
    edge_st_col: str = "StartTime_5races"
    edge_rating_col: str = "glicko2_rating_waku"
    edge_idx_st: int = field(init=False)
    edge_idx_rating: int = field(init=False)

    lane_emb_dim: int = 8
    edge_emb_dim: int = 8
    hidden_dim: int = 32
    embed_dim: int = 32
    heads: int = 4
    max_lane_dist: int = 2
    dropout: float = 0.15

    def __post_init__(self) -> None:
        self.edge_idx_st = self.feature_cols.index(self.edge_st_col)
        self.edge_idx_rating = self.feature_cols.index(self.edge_rating_col)

CFG = Config()

def seed_everything(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# 乱数を固定する(再現性のため)
seed_everything(CFG.seed)
# =========================
# Features(GNNで埋め込みを作成するために必要な特徴量)
# =========================
FEATURE_COLS = CFG.feature_cols

# エッジの特徴量とエッジインデックス
EDGE_ST_COL = CFG.edge_st_col
EDGE_RATING_COL = CFG.edge_rating_col
EDGE_IDX_ST = CFG.edge_idx_st
EDGE_IDX_RATING = CFG.edge_idx_rating

このモジュールがやっていることは下記の図の通り。

```Python
           CSV
            │
            ▼
  RaceDataProcessor(配列作成)
            │
            ▼
    RaceGraphDataset(Graph化)
            │
            ▼
        DataLoader
            │
            ▼
     TenkaiGATEncoder(GNNモデル)
            │
            ▼
 RaceEmbeddingBuilder(展開embedding)
            │
            ▼
        CSV出力
```

後続の処理を動作させるためにクラスを定義しておく。

<div align='center'><img src='./feature_engineering_emb.png' width='1200'></div>

In [ ]:
# =========================
# Data prep
# =========================
class RaceDataProcessor:
    def __init__(
        self,
        feature_cols: list[str] | None = None,
        num_boats: int = CFG.num_boats,
        max_lane_dist: int = CFG.max_lane_dist,
    ):
        self.feature_cols = feature_cols
        self.feature_cols = (
            list(feature_cols) if feature_cols is not None else list(CFG.feature_cols)
        )
        self.num_boats = num_boats
        self.max_lane_dist = max_lane_dist

    def load_race_dataframe(self, csv_path: str | Path) -> pl.DataFrame:
        return pl.read_csv(csv_path, try_parse_dates=True).sort(
            ["Date", "Venue", "Round", "Teiban"],
            descending=[False, False, False, False],
        )

    def build_race_arrays(
        self,
        df: pl.DataFrame,
        with_label: bool = True,
    ) -> tuple[
        np.ndarray,
        np.ndarray,
        np.ndarray | None,
        np.ndarray | None,
        np.ndarray,
        np.ndarray,
    ]:
        # 6艇揃っているレースだけ残す
        valid_race_ids = (
            df.group_by("RaceId", maintain_order=True)
            .len()
            .filter(pl.col("len") == self.num_boats)
            .select("RaceId")
        )
        df = df.join(valid_race_ids, on="RaceId", how="inner")

        select_cols = ["RaceId", "Date", "Teiban"] + self.feature_cols
        if with_label:
            select_cols = [
                "RaceId",
                "Date",
                "Teiban",
                "ReverseRanking",
            ] + self.feature_cols

        # 特徴量の欠損補完は race mean
        df2 = (
            df.sort(["RaceId", "Teiban"])
            .select(select_cols)
            .with_columns(
                [
                    pl.col(col).fill_null(pl.col(col).over("RaceId").mean()).alias(col)
                    for col in self.feature_cols
                ]
            )
        )

        # RaceId 単位の艇番順を保証
        counts = df2.group_by("RaceId").len().sort("RaceId")
        assert (
            counts["len"].to_list() == [self.num_boats] * counts.height
        ), "6艇でないレースが混ざっています"

        X = df2.select(self.feature_cols).to_numpy().astype(np.float32)
        lane = df2["Teiban"].to_numpy().astype(np.int64) - 1

        num_races = X.shape[0] // self.num_boats
        X_race = X.reshape(num_races, self.num_boats, len(self.feature_cols))
        lane_race = lane.reshape(num_races, self.num_boats)

        ranks_race = None
        y = None
        if with_label:
            # ReverseRanking: 6(1着) ... 1(6着) を 0(1着) ... 5(6着) に変換
            ranks = (
                self.num_boats - df2["ReverseRanking"].to_numpy().astype(np.int64)
            ).reshape(num_races, self.num_boats)
            ranks_race = ranks
            y = ranks_race.argmin(axis=1).astype(np.int64)

        race_ids = df2["RaceId"].to_numpy()[:: self.num_boats]
        race_dates = df2["Date"].to_numpy()[:: self.num_boats]
        return X_race, lane_race, ranks_race, y, race_ids, race_dates

    def build_graph(
        self,
        x_race: np.ndarray,
        lane_race: np.ndarray,
        ranks_race: np.ndarray | None,
    ) -> Data:
        x = torch.tensor(x_race, dtype=torch.float32)
        lane = torch.tensor(lane_race, dtype=torch.long)
        num_boats = x.size(0)

        idx = torch.arange(num_boats, dtype=torch.long)
        dist = (idx[:, None] - idx[None, :]).abs()
        mask = dist <= self.max_lane_dist
        src_idx, dst_idx = mask.nonzero(as_tuple=True)
        edge_index = torch.stack([src_idx, dst_idx], dim=0)
        edge_dist_idx = dist[src_idx, dst_idx].clamp(max=self.max_lane_dist).long()

        lane_dist = dist[src_idx, dst_idx].float().unsqueeze(-1) / max(
            float(self.max_lane_dist), 1.0
        )
        st = x[:, EDGE_IDX_ST]
        rating = x[:, EDGE_IDX_RATING]
        st_diff = (st[src_idx] - st[dst_idx]).abs().unsqueeze(-1)
        rating_diff = (rating[src_idx] - rating[dst_idx]).abs().unsqueeze(-1)
        edge_attr = torch.cat([lane_dist, st_diff, rating_diff], dim=1)

        graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lane=lane)
        graph.edge_dist_idx = edge_dist_idx
        if ranks_race is not None:
            graph.ranks = torch.tensor(ranks_race, dtype=torch.long)
        return graph


class RaceGraphDataset(Dataset):
    def __init__(
        self,
        data_processor: RaceDataProcessor,
        X_race: np.ndarray,
        lane_race: np.ndarray,
        ranks_race: np.ndarray | None = None,
    ):
        self.graphs = [
            data_processor.build_graph(
                x_race=X_race[i],
                lane_race=lane_race[i],
                ranks_race=ranks_race[i] if ranks_race is not None else None,
            )
            for i in range(len(X_race))
        ]

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        return self.graphs[idx]


# =========================
# Model
# =========================
class TenkaiGATEncoder(nn.Module):
    def __init__(
        self,
        cont_dim: int,
        config: Config,
        lane_emb_dim: int = 8,
        hidden_dim: int = 32,
        embed_dim: int = 32,
        heads: int = 4,
        edge_emb_dim: int = 8,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.config = config
        self.num_boats = self.config.num_boats

        self.lane_embed = nn.Embedding(self.num_boats, lane_emb_dim)
        self.input_norm = nn.LayerNorm(cont_dim + lane_emb_dim)
        self.in_proj = nn.Linear(cont_dim + lane_emb_dim, hidden_dim)

        self.edge_embed1 = nn.Embedding(self.config.max_lane_dist + 1, edge_emb_dim)
        self.gat1 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            heads=heads,
            concat=True,
            dropout=dropout,
            edge_dim=edge_emb_dim + 3,
            add_self_loops=False,
        )
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.skip1 = nn.Linear(hidden_dim, hidden_dim * heads)
        self.gat1_out_dropout = nn.Dropout(dropout)

        self.edge_embed2 = nn.Embedding(self.config.max_lane_dist + 1, edge_emb_dim)
        self.gat2 = GATv2Conv(
            in_channels=hidden_dim * heads,
            out_channels=embed_dim // heads,
            heads=heads,
            concat=True,
            dropout=dropout,
            edge_dim=edge_emb_dim + 3,
            add_self_loops=False,
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.skip2 = nn.Linear(hidden_dim * heads, embed_dim)
        self.gat2_out_dropout = nn.Dropout(dropout)
        self.attn_dropout = nn.Dropout(dropout)
        self.gat_dropout = nn.Dropout(dropout)

        self.score_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, data: Data) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        lane_emb = self.lane_embed(data.lane)
        x = torch.cat([data.x, lane_emb], dim=-1)
        x = self.input_norm(x)
        x0 = self.in_proj(x)

        edge_lane_emb1 = self.edge_embed1(data.edge_dist_idx)
        edge_attr1 = torch.cat([edge_lane_emb1, data.edge_attr], dim=-1)
        h1_raw = self.gat1(x0, data.edge_index, edge_attr1)
        h1_raw = self.gat1_out_dropout(h1_raw)
        h1 = self.norm1(F.relu(h1_raw + self.skip1(x0)))
        h1 = self.gat_dropout(h1)

        edge_lane_emb2 = self.edge_embed2(data.edge_dist_idx)
        edge_attr2 = torch.cat([edge_lane_emb2, data.edge_attr], dim=-1)
        h2_raw, (_, alpha2) = self.gat2(
            h1,
            data.edge_index,
            edge_attr2,
            return_attention_weights=True,
        )
        h2_raw = self.gat2_out_dropout(h2_raw)
        z = self.norm2(F.relu(h2_raw + self.skip2(h1)))
        z = self.gat_dropout(z)

        scores = self.score_head(z).squeeze(-1)
        alpha2_mean = alpha2.mean(dim=1)
        alpha2_dense = to_dense_adj(
            edge_index=data.edge_index,
            batch=data.batch,
            edge_attr=alpha2_mean,
            max_num_nodes=self.num_boats,
        )
        alpha2_dense = self.attn_dropout(alpha2_dense)
        return z, scores, alpha2_dense


# =========================
# Race embedding builder
# =========================
class RaceEmbeddingBuilder:
    def build(
        self,
        z: torch.Tensor,
        scores: torch.Tensor,
        alpha2: torch.Tensor,
    ) -> torch.Tensor:
        """
        展開埋め込みを race-level に集約する。

        ただ平均するだけではなく、
        - mean node embedding
        - max node embedding
        - sorted score
        - score gap
        - attention interaction summary
        を結合する。
        """
        # z: (B,6,E), scores: (B,6), alpha2: (B,6,6)
        mean_z = z.mean(dim=1)  # (B,E)
        max_z = z.max(dim=1).values  # (B,E)

        sorted_scores, _ = torch.sort(scores, dim=1, descending=True)  # (B,6)
        score_gaps = sorted_scores[:, :-1] - sorted_scores[:, 1:]  # (B,5)
        pairwise_diff = (
            (z.unsqueeze(2) - z.unsqueeze(1)).abs().mean(dim=(1, 2))
        )  # (B,E)
        row_strength = alpha2.sum(dim=-1)  # (B,6)
        col_strength = alpha2.sum(dim=-2)  # (B,6)

        return torch.cat(
            [
                mean_z,
                max_z,
                sorted_scores,
                score_gaps,
                pairwise_diff,
                row_strength,
                col_strength,
            ],
            dim=1,
        )


# =========================
# Runner
# =========================
class TenkaiEmbeddingTrainer:
    def __init__(
        self,
        config: Config = CFG,
        feature_cols: tuple[str, ...] | None = None,
        device: torch.device = DEVICE,
    ):
        self.config = config
        self.feature_cols = (
            list(feature_cols) if feature_cols is not None else list(FEATURE_COLS)
        )
        self.device = device
        self.data_processor = RaceDataProcessor(
            feature_cols=self.feature_cols,
        )
        self.embedding_builder = RaceEmbeddingBuilder()

    def build_model(self) -> TenkaiGATEncoder:
        model = TenkaiGATEncoder(
            cont_dim=len(self.feature_cols),
            config=self.config,
            lane_emb_dim=self.config.lane_emb_dim,
            hidden_dim=self.config.hidden_dim,
            embed_dim=self.config.embed_dim,
            heads=self.config.heads,
            edge_emb_dim=self.config.edge_emb_dim,
            dropout=self.config.dropout,
        ).to(self.device)
        return model

    def build_time_split_indices(
        self,
        race_dates: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        race_meta = pl.DataFrame(
            {"idx": np.arange(len(race_dates), dtype=np.int64), "Date": race_dates}
        )
        test_start_date = race_meta.select(
            pl.col("Date").max() - pl.duration(days=30)
        ).to_series()[0]
        days_before = test_start_date - datetime.timedelta(days=30)

        train_idx = race_meta.filter(pl.col("Date") < days_before)["idx"].to_numpy()
        val_idx = race_meta.filter(
            (pl.col("Date") >= days_before) & (pl.col("Date") < test_start_date)
        )["idx"].to_numpy()
        test_idx = race_meta.filter(pl.col("Date") >= test_start_date)["idx"].to_numpy()
        return train_idx, val_idx, test_idx

    def _accuracy_from_logits(self, logits: torch.Tensor, y: torch.Tensor) -> float:
        pred = logits.argmax(dim=1)
        return (pred == y).float().mean().item()

    def _plackett_luce_loss(
        self, scores: torch.Tensor, ranks: torch.Tensor
    ) -> torch.Tensor:
        """
        scores: (B, N) 各艇スコア（大きいほど上位）
        ranks : (B, N) 0 が1着、N-1 が最下位
        """
        order = torch.argsort(ranks, dim=1, descending=False)
        ordered_scores = scores.gather(1, order)
        n = ordered_scores.size(1)

        loss = torch.zeros(ordered_scores.size(0), device=scores.device)
        for k in range(n):
            denom = torch.logsumexp(ordered_scores[:, k:], dim=1)
            loss = loss + (denom - ordered_scores[:, k])
        return loss.mean()

    def _run_epoch(self, model, loader, optimizer=None):
        is_train = optimizer is not None
        if is_train:
            model.train()
        else:
            model.eval()

        total_loss = 0.0
        total_acc = 0.0
        total_n = 0

        for batch in loader:
            batch = batch.to(self.device)
            ranks_batch = batch.ranks.view(-1, self.config.num_boats)
            y_batch = ranks_batch.argmin(dim=1)

            with torch.set_grad_enabled(is_train):
                _, scores, _ = model(batch)
                scores_race = scores.view(-1, self.config.num_boats)
                loss = self._plackett_luce_loss(scores_race, ranks_batch)

                if is_train:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

            batch_n = y_batch.size(0)
            total_loss += loss.item() * batch_n
            total_acc += self._accuracy_from_logits(scores_race, y_batch) * batch_n
            total_n += batch_n

        return total_loss / total_n, total_acc / total_n

    def train_model(
        self,
        X_race: np.ndarray,
        lane_race: np.ndarray,
        ranks_race: np.ndarray,
        race_dates: np.ndarray,
    ) -> TenkaiGATEncoder:
        train_idx, val_idx, test_idx = self.build_time_split_indices(race_dates)
        if len(train_idx) == 0 or len(val_idx) == 0:
            raise ValueError(
                "train/valid が空です。データ期間と日付分割条件を確認してください。"
            )
        print(
            f"split sizes: train={len(train_idx)} valid={len(val_idx)} test={len(test_idx)}"
        )

        train_ds = RaceGraphDataset(
            self.data_processor,
            X_race[train_idx],
            lane_race[train_idx],
            ranks_race[train_idx],
        )
        val_ds = RaceGraphDataset(
            self.data_processor,
            X_race[val_idx],
            lane_race[val_idx],
            ranks_race[val_idx],
        )
        train_loader = DataLoader(
            train_ds, batch_size=self.config.batch_size, shuffle=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=self.config.batch_size, shuffle=False
        )
        model = self.build_model()

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.config.lr,
            weight_decay=self.config.weight_decay,
        )

        best_val_loss = math.inf
        best_state = None

        for epoch in range(1, self.config.epochs + 1):
            train_loss, train_acc = self._run_epoch(model, train_loader, optimizer)
            val_loss, val_acc = self._run_epoch(model, val_loader, optimizer=None)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {
                    k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                }

            print(
                f"epoch={epoch:02d} "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
            )

        if best_state is not None:
            model.load_state_dict(best_state)
        return model

    def pred_embeddings(
        self,
        model: TenkaiGATEncoder,
        X_race: np.ndarray,
        lane_race: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        full_ds = RaceGraphDataset(
            self.data_processor,
            X_race,
            lane_race,
            ranks_race=None,
        )
        full_loader = DataLoader(
            full_ds, batch_size=self.config.batch_size, shuffle=False
        )
        model.eval()
        race_emb_list = []
        score_list = []
        winner_pred_list = []

        with torch.no_grad():
            for batch in full_loader:
                batch = batch.to(self.device)
                z, scores, alpha2 = model(batch)
                z_race = z.view(-1, self.config.num_boats, z.size(-1))
                scores_race = scores.view(-1, self.config.num_boats)
                race_emb = self.embedding_builder.build(z_race, scores_race, alpha2)

                race_emb_list.append(race_emb.cpu())
                score_list.append(scores_race.cpu())
                winner_pred_list.append(scores_race.argmax(dim=1).cpu())

        race_embedding = torch.cat(race_emb_list, dim=0).numpy()
        winner_scores = torch.cat(score_list, dim=0).numpy()
        winner_pred = torch.cat(winner_pred_list, dim=0).numpy()
        return race_embedding, winner_scores, winner_pred

    def build_embedding_dataframe(
        self,
        race_ids: np.ndarray,
        race_embedding: np.ndarray,
        winner_scores: np.ndarray,
        winner_pred: np.ndarray,
        y: np.ndarray | None = None,
    ) -> pl.DataFrame:
        emb_cols = [f"tenkai_emb{i}" for i in range(race_embedding.shape[1])]
        score_cols = [f"winner_score_{i+1}" for i in range(self.config.num_boats)]
        payload: dict[str, Any] = {
            "RaceId": race_ids,
            **{col: race_embedding[:, i] for i, col in enumerate(emb_cols)},
            **{col: winner_scores[:, i] for i, col in enumerate(score_cols)},
            "winner_pred": winner_pred + 1,
        }
        if y is not None:
            payload["winner_true"] = y + 1

        return pl.DataFrame(payload).sort("RaceId")

    def save_model_checkpoint(self, model: TenkaiGATEncoder) -> None:
        model_file = Path(self.config.model_path)
        model_file.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": model.state_dict()}, model_file)
        print(f"saved model: {model_file}")

    def load_model_checkpoint(self) -> TenkaiGATEncoder:
        model = self.build_model()
        checkpoint = torch.load(self.config.model_path, map_location=self.device)
        model.load_state_dict(checkpoint["state_dict"])
        return model

    def run_train_mode(self) -> TenkaiGATEncoder:
        train_df = self.data_processor.load_race_dataframe(self.config.train_csv_path)
        X_race, lane_race, ranks_race, y, race_ids, race_dates = (
            self.data_processor.build_race_arrays(
                df=train_df,
                with_label=True,
            )
        )
        assert ranks_race is not None
        assert y is not None

        model = self.train_model(X_race, lane_race, ranks_race, race_dates)
        self.save_model_checkpoint(model)

        race_embedding, winner_scores, winner_pred = self.pred_embeddings(
            model, X_race, lane_race
        )
        emb_df = self.build_embedding_dataframe(
            race_ids=race_ids,
            race_embedding=race_embedding,
            winner_scores=winner_scores,
            winner_pred=winner_pred,
            y=y,
        )
        out_path = Path(self.config.train_emb_out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        emb_df.write_csv(out_path)
        print(f"saved train embedding: {out_path}")
        print(emb_df.head())
        return model

    def run_pred_mode(self, model: TenkaiGATEncoder | None = None) -> None:
        if model is None:
            model = self.load_model_checkpoint()

        pred_df = self.data_processor.load_race_dataframe(self.config.pred_csv_path)
        X_race, lane_race, _, _, race_ids, _ = self.data_processor.build_race_arrays(
            df=pred_df, with_label=False
        )
        race_embedding, winner_scores, winner_pred = self.pred_embeddings(
            model, X_race, lane_race
        )
        emb_df = self.build_embedding_dataframe(
            race_ids=race_ids,
            race_embedding=race_embedding,
            winner_scores=winner_scores,
            winner_pred=winner_pred,
            y=None,
        )
        out_path = Path(self.config.pred_emb_out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        emb_df.write_csv(out_path)
        print(f"saved prediction embedding: {out_path}")
        print(emb_df.head())

    def run(self, mode: str) -> None:
        if mode == "train":
            self.run_train_mode()
        elif mode == "pred":
            self.run_pred_mode(model=None)
        else:
            trained_model = self.run_train_mode()
            self.run_pred_mode(model=trained_model)

## `RaceDataProcessor`クラス

まずは`RaceDataProcessor`クラスを利用して、読み込んだデータに対して処理を行う。`load_race_dataframe`メソッドではデータの読み込みを行い、後続の処理のためにソートする。

In [63]:
data_processor = RaceDataProcessor()
train_df = data_processor.load_race_dataframe(CFG.train_csv_path)

In [64]:
train_df.shape

(615534, 189)

`build_race_arrays`メソッドはグラフデータへの変換前の配列が不整合を起こさないように、1レース6レコード、使用する特徴量は欠損値がないかを確認し、問題のないレースを配列化する。

In [65]:
X_race, lane_race, ranks_race, y, race_ids, race_dates = (
    data_processor.build_race_arrays(
        df=train_df,
        with_label=True,
    )
)

In [71]:
print(f"X_race.shape: {X_race.shape}")
print(f"lane_race.shape: {np.array(lane_race).shape}")
print(f"ranks_race.shape: {np.array(ranks_race).shape}")
print(f"y.shape: {np.array(y).shape}")
print(f"race_ids.shape: {np.array(race_ids).shape}")
print(f"race_dates.shape: {np.array(race_dates).shape}")

X_race.shape: (102589, 6, 22)
lane_race.shape: (102589, 6)
ranks_race.shape: (102589, 6)
y.shape: (102589,)
race_ids.shape: (102589,)
race_dates.shape: (102589,)


In [ ]:
train_df.filter(pl.col("RaceId") == "20240314_03_01").select(
    ["RaceId", "Teiban", "PlayerNo", "ReverseRanking"]
).with_columns(7 - pl.col("ReverseRanking").cast(pl.Int32).alias("Ranking"))

RaceId,Teiban,PlayerNo,ReverseRanking,literal
str,i64,i64,i64,i32
"""20240314_03_01""",1,2014,5,2
"""20240314_03_01""",2,3776,6,1
"""20240314_03_01""",3,4603,4,3
"""20240314_03_01""",4,4453,3,4
"""20240314_03_01""",5,5266,1,6
"""20240314_03_01""",6,5298,2,5


上記のデータフレームと対応させるとわかりよい。

- 今回のデータは102589レース分が対象。
- `X_race`はレースごとに6艇分の22個の特徴量が保存されている。
- `lane_race`は各レースの艇の枠番を表す。0indexに修正。
- `ranks_race`は各レースの挺が何着だったかを示す。`ReverseRanking`なので6が1着。`Ranking`に内部で戻す。
    - 今回のレースは2号艇が1着、つまり、`ReverseRanking`は6、`Ranking`は1なので`ranks_race`は0という対応関係。
- `y`は`ranks_race`の1着の配列のインデックス。
- `race_ids`はレースを識別するID
- `race_dates`はレースの開催日


In [ ]:
print("----- show first race -----")
# print("X_race:", X_race[0,:,:])
print("lane_race:", np.array(lane_race)[0, :])
print("ranks_race:", np.array(ranks_race)[0, :])
print("y:", np.array(y)[0])
print("race_ids:", np.array(race_ids)[0])
print("race_dates:", np.array(race_dates)[0])

----- show first race -----
lane_race: [0 1 2 3 4 5]
ranks_race: [1 0 2 3 5 4]
y: 1
race_ids: 20240314_03_01
race_dates: 2024-03-14


配列化されたデータは、`build_graph`メソッドでグラフデータに変換する。説明のために1つ取り出しておく。

In [ ]:
# ranks_race = ranks_race
x = torch.tensor(X_race[0], dtype=torch.float32)
lane = torch.tensor(lane_race[0], dtype=torch.long)
num_boats = x.size(0)

idx = torch.arange(num_boats, dtype=torch.long)
dist = (idx[:, None] - idx[None, :]).abs()
mask = dist <= CFG.max_lane_dist
src_idx, dst_idx = mask.nonzero(as_tuple=True)
edge_index = torch.stack([src_idx, dst_idx], dim=0)
edge_dist_idx = dist[src_idx, dst_idx].clamp(max=CFG.max_lane_dist).long()

- `idx`は、0から5までの各艇のインデックスを表すテンソル（tensor([0, 1, 2, 3, 4, 5])）。
- `dist`は、各艇同士のインデックス差（絶対値）を6×6の距離行列として表したもの。自分との距離は0、例えば1行4列なら|1-4|=3となり3つ離れていることを意味する。
- `mask`は、distの値がCFG.max_lane_dist以下である位置をTrue、そうでない所をFalseにした真偽値マスク。これにより「近い艇同士」のみを抽出。
- `src_idx`と`dst_idx`は、maskがTrueとなる各エッジの始点（src）と終点（dst）のインデックス。グラフのエッジリストとなる。
- `edge_index`は、[src_idx, dst_idx]を(2, N)の形にまとめたテンソルで、グラフのつながりを示す。
- `edge_dist_idx`は、それぞれのエッジ（src_idx, dst_idx）のdist（距離）の値を、最大値CFG.max_lane_distでクリップしたテンソル。
    - これは`edge_dist_idx`が、近接している艇同士の「レーン距離」を示すインデックステンソル。近い艇同士のみを頂点・エッジとしたグラフデータを作成。
    - tensor([
        - 1号艇: 0(1号艇), 1(2号艇), 2(3号艇)
        - 2号艇: 1(1号艇), 0(2号艇), 1(3号艇), 2(4号艇)
        - 3号艇: 2(1号艇), 1(2号艇), 0(3号艇), 1(4号艇), 2(5号艇)
        - 4号艇: 2(2号艇), 1(3号艇), 0(4号艇), 1(5号艇), 2(6号艇), 
        - 5号艇: 2(3号艇), 1(4号艇), 0(5号艇), 1(6号艇),
        - 6号艇: 2(号4艇), 1(5号艇), 0(6号艇)
    -    ])

In [89]:
print("idx:", idx, "shape:", idx.shape)
print("dist:\n", dist, "shape:", dist.shape)
print("mask:\n", mask, "shape:", mask.shape)
print("src_idx:", src_idx, "shape:", src_idx.shape)
print("dst_idx:", dst_idx, "shape:", dst_idx.shape)
print("edge_index:\n", edge_index, "shape:", edge_index.shape)
print("edge_dist_idx:", edge_dist_idx, "shape:", edge_dist_idx.shape)

idx: tensor([0, 1, 2, 3, 4, 5]) shape: torch.Size([6])
dist:
 tensor([[0, 1, 2, 3, 4, 5],
        [1, 0, 1, 2, 3, 4],
        [2, 1, 0, 1, 2, 3],
        [3, 2, 1, 0, 1, 2],
        [4, 3, 2, 1, 0, 1],
        [5, 4, 3, 2, 1, 0]]) shape: torch.Size([6, 6])
mask:
 tensor([[ True,  True,  True, False, False, False],
        [ True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True, False],
        [False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True]]) shape: torch.Size([6, 6])
src_idx: tensor([0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5]) shape: torch.Size([24])
dst_idx: tensor([0, 1, 2, 0, 1, 2, 3, 0, 1, 2, 3, 4, 1, 2, 3, 4, 5, 2, 3, 4, 5, 3, 4, 5]) shape: torch.Size([24])
edge_index:
 tensor([[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5],
        [0, 1, 2, 0, 1, 2, 3, 0, 1, 2, 3, 4, 1, 2, 3, 4, 5, 2, 3, 4, 5, 3,

ここでは、艇同士の「レーン距離」「スタートタイムの差」「評価値の差」などをグラフのエッジ特徴量（`edge_attr`）として作成。

1. `lane_dist`（レーン距離）：  
   エッジで結ばれた2艇それぞれのレーン番号の差（絶対値）を、`CFG.max_lane_dist`で正規化した値。  
   例えば、1号艇と2号艇なら「1」のように、どれくらいレーンが離れているかを表し、遠い艇同士では値が大きく、近い艇同士では値が小さくなる。

2. `st_diff`（スタートタイム差）：  
   横の艇のスタートタイム（ST, Start Timing）の絶対値差。  
   各艇がスタートでどのぐらいタイミングに違いがあったかを示す指標で、大きいほどスタートの差が大きくなる。

3. `rating_diff`（評価値の差）：  
   横の艇それぞれの枠×選手グリコ2レーティングの絶対値差。  
   評価値（例えば過去成績等を元にしたスコア）が似ていれば値が小さく、異なるほど大きくなる。

これら3つはそれぞれ (エッジ数, 1) 形状のテンソル（縦ベクトル）として計算されており、`torch.cat` で次元1（横方向、列方向）に連結することで(エッジ数, 3) の行列となる。こうして作成した`edge_attr`は、グラフニューラルネットワークでのアテンション学習時、「近い艇同士がどれくらい特徴が似ているか／異なるか」「レーンが離れるごとにどう関係が変わるか」など艇同士の関係性把握に利用する。

In [ ]:
lane_dist = dist[src_idx, dst_idx].float().unsqueeze(-1) / max(
    float(CFG.max_lane_dist), 1.0
)
st = x[:, EDGE_IDX_ST]
rating = x[:, EDGE_IDX_RATING]
st_diff = (st[src_idx] - st[dst_idx]).abs().unsqueeze(-1)
rating_diff = (rating[src_idx] - rating[dst_idx]).abs().unsqueeze(-1)
edge_attr = torch.cat([lane_dist, st_diff, rating_diff], dim=1)

In [99]:
print("lane_dist:", lane_dist, "shape:", lane_dist.shape)
print("st:", st, "shape:", st.shape)
print("st_diff:", st_diff.T, "shape:", st_diff.shape)
print("rating:", rating, "shape:", rating.shape)
print("rating_diff:", rating_diff.T, "shape:", rating_diff.shape)
print("edge_index:\n", edge_index, "shape:", edge_index.shape)
print("edge_attr:\n", edge_attr, "shape:", edge_attr.shape)

lane_dist: tensor([[0.0000],
        [0.5000],
        [1.0000],
        [0.5000],
        [0.0000],
        [0.5000],
        [1.0000],
        [1.0000],
        [0.5000],
        [0.0000],
        [0.5000],
        [1.0000],
        [1.0000],
        [0.5000],
        [0.0000],
        [0.5000],
        [1.0000],
        [1.0000],
        [0.5000],
        [0.0000],
        [0.5000],
        [1.0000],
        [0.5000],
        [0.0000]]) shape: torch.Size([24, 1])
st: tensor([0.1500, 0.1740, 0.1560, 0.1900, 0.1560, 0.1960]) shape: torch.Size([6])
st_diff: tensor([[0.0000, 0.0240, 0.0060, 0.0240, 0.0000, 0.0180, 0.0160, 0.0060, 0.0180,
         0.0000, 0.0340, 0.0000, 0.0160, 0.0340, 0.0000, 0.0340, 0.0060, 0.0000,
         0.0340, 0.0000, 0.0400, 0.0060, 0.0400, 0.0000]]) shape: torch.Size([24, 1])
rating: tensor([1369.4302, 1482.6660, 1556.4442, 1232.6460, 1023.4890, 1241.0061]) shape: torch.Size([6])
rating_diff: tensor([[  0.0000, 113.2358, 187.0140, 113.2358,   0.0000,  73.7782, 

ここまでテンソル化したものをPyTorchGで利用するためにグラフデータに変換する。

In [100]:
graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lane=lane)
graph.edge_dist_idx = edge_dist_idx
graph.ranks = torch.tensor(ranks_race, dtype=torch.long)
graph

Data(x=[6, 22], edge_index=[2, 24], edge_attr=[24, 3], lane=[6], edge_dist_idx=[24], ranks=[102589, 6])

`RaceGraphDataset`クラスから`RaceDataProcessor`クラスの`build_graph`メソッドを呼び出すことで、レース分のグラフデータをまとめる。`build_graph`メソッドは各レースをグラフデータにするもので、それらをまとめるのが`RaceGraphDataset`クラスのような役割。

In [ ]:
graphs = [
    data_processor.build_graph(
        x_race=X_race[i],
        lane_race=lane_race[i],
        ranks_race=ranks_race[i] if ranks_race is not None else None,
    )
    for i in range(len(X_race))
]

# len(graphs): 102589

In [111]:
g0 = graphs[0]
print(type(g0))
print(g0)
print("x shape:", g0.x.shape)
print("edge_index shape:", g0.edge_index.shape)
print("edge_attr shape:", g0.edge_attr.shape)
print("lane shape:", g0.lane.shape)
print("has ranks:", hasattr(g0, "ranks"))
if hasattr(g0, "ranks"):
    print("ranks shape:", g0.ranks.shape)

<class 'torch_geometric.data.data.Data'>
Data(x=[6, 22], edge_index=[2, 24], edge_attr=[24, 3], lane=[6], edge_dist_idx=[24], ranks=[6])
x shape: torch.Size([6, 22])
edge_index shape: torch.Size([2, 24])
edge_attr shape: torch.Size([24, 3])
lane shape: torch.Size([6])
has ranks: True
ranks shape: torch.Size([6])


## `TenkaiEmbeddingTrainer`クラス

`TenkaiEmbeddingTrainer`クラスは、グラフデータを用いて埋め込み表現（エンベディング）を学習するためのクラス。主な処理内容は以下の通り。

1. グラフデータセットやインデックスの準備、データローダーの構築など、学習や検証に使用するデータ分割。
2. `build_time_split_indices`メソッドによって、レース日時情報をもとに学習・検証・テスト用のインデックスを作成。
3. 構成情報（特徴量数や隠れ層次元など）に基づき、GATv2をベースにしたエンコーダーモデル（`TenkaiGATEncoder`）を構築。
4. モデルの学習ループ処理では、各バッチに対して損失計算と勾配更新を行い、検証データでの性能評価も実施。
5. 学習済みモデルの保存や、必要に応じて予測や特徴量抽出も行う。

`build_time_split_indices`メソッドはデータを3つの区切りで、学習、検証、テストに分割するためのインデックスを返す。これは、埋め込み特徴量を加えたLightGBMが学習する際と同じデータの分割方法。

In [112]:
tet = TenkaiEmbeddingTrainer()
# model = tet.train_model(X_race, lane_race, ranks_race, race_dates)
train_idx, val_idx, test_idx = tet.build_time_split_indices(race_dates)

In [ ]:
train_ds = RaceGraphDataset(
    RaceDataProcessor(CFG.feature_cols),
    X_race[train_idx],
    lane_race[train_idx],
    ranks_race[train_idx],
)
val_ds = RaceGraphDataset(
    RaceDataProcessor(CFG.feature_cols),
    X_race[val_idx],
    lane_race[val_idx],
    ranks_race[val_idx],
)

訓練、検証に分けたグラフデータをミニバッチ学習ができるように`DataLoader`で処理する。

In [ ]:
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True) # fmt: skip
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False) # fmt:skip

## `TenkaiGATEncoder`クラス

`TenkaiGATEncoder`クラスは、各艇の特徴量やレーン情報、エッジ情報をもとに、GATv2で艇ごとの埋め込み表現を生成するエンコーダ。ノードの連続値特徴・埋め込み・レーン埋め込み、エッジ特徴に対応し、複数ヘッド/層のGATConvを使って艇間関係・展開を学習する。

In [ ]:
model = TenkaiGATEncoder(
    cont_dim=len(CFG.feature_cols),
    config=CFG,
    lane_emb_dim=CFG.lane_emb_dim,
    hidden_dim=CFG.hidden_dim,
    embed_dim=CFG.embed_dim,
    heads=CFG.heads,
    edge_emb_dim=CFG.edge_emb_dim,
    dropout=CFG.dropout,
).to(DEVICE)

このモデルの中心は GATv2Convで、「どの艇同士の関係を重視するか」をアテンション機構で学習する。

- 1) 入力層
    - lane_embed: Embedding(6, 8)
    6レーンを、8次元ベクトルに変換。
    - input_norm: LayerNorm(30)
        - 22個の特徴量に加え、lane_embedを結合して[6, 30]の状態。30次元の入力特徴量を正規化して学習を安定化。
    - in_proj: Linear(30 → 32)
        - 入力特徴を30次元から32次元へ写像（内部表現の初期化）。
        - GATv2が32で受けるので、拡張しておく。

- 2) グラフ畳み込み（1層目）
    - edge_embed1: Embedding(3, 8)
    3種類のエッジ属性([lane_dist, st_diff, rating_diff])を8次元ベクトルへ変換。
    - gat1: GATv2Conv(32 → 32, heads=4)
        - 4ヘッドアテンションで近傍ノード情報を集約。出力は実質 32×4=128 次元。
        - 必ずしも関係するわけではないが、エッジ属性にあわせて=1して4にした
        - head1 → ST勝負、head2 → rating勝負、head3 → コース距離、head4 → 展開
            - これは想像であって、実際はどうなっているかはわからない
    - norm1: LayerNorm(128)
    1段目出力を正規化。
    - skip1: Linear(32 → 128)
    残差接続用に入力側も128次元へ合わせる。
    - gat1_out_dropout: Dropout(0.15)
    過学習を防ぐため15%ドロップアウト。

- 3) グラフ畳み込み（2層目）
    - edge_embed2: Embedding(3, 8)
    2段目用のエッジ埋め込み。
    - gat2: GATv2Conv(128 → 8, heads=4)
    4ヘッドアテンションで集約。出力は 8×4=32 次元。
    - norm2: LayerNorm(32)
    2段目出力を正規化。
    - skip2: Linear(128 → 32)
    残差接続の次元合わせ。
    - gat2_out_dropout: Dropout(0.15)
    2段目出力にもドロップアウト。

- 4) ドロップアウト
    - attn_dropout: Dropout(0.15)
    アテンションの重み周辺の正則化用。
    - gat_dropout: Dropout(0.15)
    GAT内部表現の正則化用。

- 5) 出力層（score_head）
    - LayerNorm(32)
    - Linear(32 → 32)
    - ReLU
    - Dropout(0.15)
    - Linear(32 → 1)
    - 最終的に、各ノードに対して1つのスコア値を出力。

```Python
node features
(6,22)
     │
lane embedding
(6,8)
     │
concat
(6,30)
     │
LayerNorm
     │
Linear(30→32)
     │
(6,32)
     │
GATv2Conv
(32→32, heads=4)
     │
(6,128)
     │
Residual + Norm
     │
(6,128)
     │
GATv2Conv
(128→8, heads=4)
     │
(6,32)
     │
Residual + Norm
     │
node embedding
(6,32)
     │
score_head
     │
scores
(6)
```

In [117]:
model

TenkaiGATEncoder(
  (lane_embed): Embedding(6, 8)
  (input_norm): LayerNorm((30,), eps=1e-05, elementwise_affine=True)
  (in_proj): Linear(in_features=30, out_features=32, bias=True)
  (edge_embed1): Embedding(3, 8)
  (gat1): GATv2Conv(32, 32, heads=4)
  (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (skip1): Linear(in_features=32, out_features=128, bias=True)
  (gat1_out_dropout): Dropout(p=0.15, inplace=False)
  (edge_embed2): Embedding(3, 8)
  (gat2): GATv2Conv(128, 8, heads=4)
  (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (skip2): Linear(in_features=128, out_features=32, bias=True)
  (gat2_out_dropout): Dropout(p=0.15, inplace=False)
  (attn_dropout): Dropout(p=0.15, inplace=False)
  (gat_dropout): Dropout(p=0.15, inplace=False)
  (score_head): Sequential(
    (0): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=32, out_features=32, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.15, inplace=False)
    (4): Line

ここではAdam+weight decay（L2正則化）を組み合わせた optimizer を利用。また、これは Early stopping用の変数を用意。

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)
best_val_loss = math.inf
best_state = None

`_accuracy_from_logits`は、モデルが計算するスコアが高いインデックスを計算し、最初に定義したレースで誰が1着になるかを示したインデックス`y`と比較して正解かどうかを判定。

In [123]:
def _accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    pred = logits.argmax(dim=1)
    return (pred == y).float().mean().item()

`_plackett_luce_loss`は1号艇から順にスコアを並べる必要がある。

- `scores = [2,1,0]`
- `ranks  = [1,0,2]`

| 艇   | score |rank |
| --- | ----- |----- |
| 1号艇 | 2.1   |2着
| 2号艇 | 1.4   |1着
| 3号艇 | 0.9   |3着


があったときに、

- `order = [1,0,2]`
    - `argsort`で`rank`の小さい順にインデックスを並べる
- `ordered_scores = [1, 2, 0]`
    - `gather`で`order` の順番で `scores` を取り出す

| 順位 | 艇   | score |
| -- | --- | ----- |
| 1着 | 2号艇 | 1.4   |
| 2着 | 1号艇 | 2.1   |
| 3着 | 3号艇 | 0.9   |

と並び替えることで、Plackett-Luceの計算式の通り、1着から順に計算して損失を求め、正しい順位ほどスコアが高くなるように学習する。このあたりの詳細は1つ前の`note_PyTorch07`が参考になる。

In [ ]:
def _plackett_luce_loss(scores: torch.Tensor, ranks: torch.Tensor) -> torch.Tensor:
    """
    scores: (B, N) 各艇スコア（大きいほど上位）
    ranks : (B, N) 0 が1着、N-1 が最下位
    """
    order = torch.argsort(ranks, dim=1, descending=False)
    ordered_scores = scores.gather(1, order)
    n = ordered_scores.size(1)
    loss = torch.zeros(ordered_scores.size(0), device=scores.device)
    for k in range(n):
        denom = torch.logsumexp(ordered_scores[:, k:], dim=1)
        loss = loss + (denom - ordered_scores[:, k])
    return loss.mean()

`_run_epoch`は定義したモデルをもとに学習を行う。optimizerがあれば学習、なければ検証となる。`batch_size = 256`の場合、1レース = 6艇なので、1 batch の中には256レース分 × 6艇 = 1536ノード入ることになる。

`batch.ranks =[1,0,2,3,5,4, 0,2,1,3,4,5, 2,0,1,4,3,5]`はこのような形式なので、`view`でレースごとに6個の着順が取れるように変換。つまり18個あれば、[3, 6]となるように変換。その変換後のデータをもとに`y_batch`では正解ラベルを取得する。

`batch_n`はこの batch に何レース入っていたかを計算する。batch_size=256 なら普通は 256、最後の batch だけ 100 などはありうるので毎回取る。

`total_loss += loss.item() * batch_n`は少しややこしい。`loss` はその batch の平均 `loss`。最後に epoch 全体の平均を出したいので、まず合計`loss`を計算してから重み付きで平均する。

- batch1: loss=0.8, 256レース
- batch2: loss=1.2, 100レース

このようなケースで(0.8+1.2)/2=1とはできない。レース数が異なるので、そのため(0.8*256 + 1.2* 100)/(256 + 100) = 0.91としないといけない。

In [121]:
def _run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    if is_train:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        ranks_batch = batch.ranks.view(-1, CFG.num_boats)
        y_batch = ranks_batch.argmin(dim=1)
        with torch.set_grad_enabled(is_train):
            _, scores, _ = model(batch)
            scores_race = scores.view(-1, CFG.num_boats)
            loss = _plackett_luce_loss(scores_race, ranks_batch)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        batch_n = y_batch.size(0)
        total_loss += loss.item() * batch_n
        total_acc += _accuracy_from_logits(scores_race, y_batch) * batch_n
        total_n += batch_n
    return total_loss / total_n, total_acc / total_n

```Python
DataLoader から batch 取得
        ↓
batch を device に送る
        ↓
ranks を (B,6) に整形
        ↓
勝者ラベル y_batch を作る
        ↓
model(batch) で艇スコアを出す
        ↓
scores を (B,6) に整形
        ↓
Plackett-Luce loss を計算
        ↓
学習時だけ backward + step
        ↓
loss と accuracy を集計
        ↓
epoch 平均を返す
```

アーリーストッピングはあとで実装する。

In [ ]:
for epoch in range(1, 50):  # CFG.epochs + 1):
    train_loss, train_acc = _run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_acc = _run_epoch(model, val_loader, optimizer=None)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            k: v.detach().cpu().clone() for k, v in model.state_dict().items()
        }
    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

epoch=01 train_loss=5.8738 train_acc=0.5030 val_loss=5.8109 val_acc=0.5396
epoch=02 train_loss=5.7975 train_acc=0.5265 val_loss=5.7921 val_acc=0.5361
epoch=03 train_loss=5.7909 train_acc=0.5279 val_loss=5.7705 val_acc=0.5428
epoch=04 train_loss=5.7755 train_acc=0.5313 val_loss=5.7791 val_acc=0.5454
epoch=05 train_loss=5.7678 train_acc=0.5349 val_loss=5.7597 val_acc=0.5477
epoch=06 train_loss=5.7675 train_acc=0.5347 val_loss=5.7504 val_acc=0.5537
epoch=07 train_loss=5.7530 train_acc=0.5387 val_loss=5.7460 val_acc=0.5458
epoch=08 train_loss=5.7459 train_acc=0.5430 val_loss=5.7237 val_acc=0.5678
epoch=09 train_loss=5.7391 train_acc=0.5457 val_loss=5.7212 val_acc=0.5616
epoch=10 train_loss=5.7350 train_acc=0.5499 val_loss=5.7085 val_acc=0.5704
epoch=11 train_loss=5.7286 train_acc=0.5541 val_loss=5.7099 val_acc=0.5718
epoch=12 train_loss=5.7232 train_acc=0.5541 val_loss=5.7069 val_acc=0.5782
epoch=13 train_loss=5.7227 train_acc=0.5554 val_loss=5.7028 val_acc=0.5741
epoch=14 train_loss=5.719

`acc`が良くないかもしれないが、予測モデルではなく埋め込み生成モデルなので、精度はLightGBM側で検証する。処理の手順としては、次はベストモデルのパラメタを保存することになる。

In [126]:
if best_state is not None:
    model.load_state_dict(best_state)

In [128]:
best_state

{'lane_embed.weight': tensor([[ 5.9810, -2.1648,  5.7261, -6.2612, -2.8308, -5.7065, -4.0104, -6.1819],
         [-0.5126,  2.3541, -0.9844, -1.1036, -1.9179,  0.2908, -0.5093,  1.4930],
         [ 1.5164,  0.5527, -0.5071,  0.5848, -0.8322,  1.2760,  0.8281,  2.2061],
         [-0.7994,  1.7040, -0.4998,  3.4591,  2.3925,  1.3473,  0.8717,  0.9801],
         [-4.4516,  0.4103, -2.7175,  5.0306,  3.3669,  2.6111,  2.6697,  1.3170],
         [-5.7709,  4.9715, -5.4983,  3.7617,  2.3626,  6.7680,  2.6977,  4.0751]]),
 'input_norm.weight': tensor([0.8015, 0.7318, 0.7171, 0.6401, 0.5667, 0.8861, 0.7684, 0.8014, 0.7681,
         0.6967, 0.7666, 0.7248, 0.7773, 0.7197, 0.7271, 0.9877, 0.8181, 0.7247,
         1.2137, 1.3532, 1.1600, 0.7745, 1.2003, 0.9462, 1.1878, 1.2849, 1.0261,
         1.3112, 1.1187, 1.2207]),
 'input_norm.bias': tensor([ 0.2220,  0.2099,  0.2054, -0.3779, -0.4317,  0.1380,  0.2110,  0.2089,
          0.2136,  0.2023,  0.2210,  0.2054,  0.2189,  0.2071,  0.2083,  0.0283,

`save_model_checkpoint`メソッドはベストモデルのパラメタを書き出して、予測時に使えるようにしている。、

In [ ]:
def save_model_checkpoint(model: TenkaiGATEncoder) -> None:
    model_file = Path(CFG.model_path)
    model_file.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": model.state_dict()}, model_file)
    print(f"saved model: {model_file}")


save_model_checkpoint(model)

## `RaceEmbeddingBuilder`クラス

このクラスは埋め込みを生成するクラス。

In [ ]:
embedding_builder = RaceEmbeddingBuilder()
data_processor = RaceDataProcessor(
    feature_cols=CFG.feature_cols,
)

`pred_embeddings`は、学習済み`TenkaiGATEncoder`を使って、各レースに対する埋め込み表現と着順の予測情報を推論するメソッド。入力としてレース特徴量 `X_race` と枠番情報 `lane_race` を受け取る。

`RaceGraphDataset`クラスをランクなしで生成し、`DataLoader（shuffle=False）`でミニバッチ化。これにより入力順を維持したまま推論できる。`model.eval()`で推論モードで、勾配計算や不要な学習挙動を防ぐ。

バッチごとの学習済みモデルにデータを渡し、`z`（ノード埋め込み）、`scores`（艇ごとのスコア）`alpha2`（注意重み）などを取得。その後、`RaceEmbeddingBuilder`クラスの`build`メソッドで`z` と `scores` を「レース単位」に集約。各バッチの埋め込みを`torch.cat`で結合し、配列に変換する。これでレースごとの埋め込みを利用できる。

In [ ]:
def pred_embeddings(
    model: TenkaiGATEncoder,
    X_race: np.ndarray,
    lane_race: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    full_ds = RaceGraphDataset(
        data_processor,
        X_race,
        lane_race,
        ranks_race=None,
    )
    full_loader = DataLoader(full_ds, batch_size=CFG.batch_size, shuffle=False)
    model.eval()
    race_emb_list = []
    score_list = []
    winner_pred_list = []

    with torch.no_grad():
        for batch in full_loader:
            batch = batch.to(DEVICE)
            z, scores, alpha2 = model(batch)
            z_race = z.view(-1, CFG.num_boats, z.size(-1))
            scores_race = scores.view(-1, CFG.num_boats)
            race_emb = embedding_builder.build(z_race, scores_race, alpha2)

            race_emb_list.append(race_emb.cpu())
            score_list.append(scores_race.cpu())
            winner_pred_list.append(scores_race.argmax(dim=1).cpu())

    race_embedding = torch.cat(race_emb_list, dim=0).numpy()
    winner_scores = torch.cat(score_list, dim=0).numpy()
    winner_pred = torch.cat(winner_pred_list, dim=0).numpy()
    return race_embedding, winner_scores, winner_pred


race_embedding, winner_scores, winner_pred = pred_embeddings(model, X_race, lane_race) # fmt: skip
race_embedding[0], winner_scores[0], winner_pred[0:10]

(array([-0.4266279 ,  0.4372305 , -0.43041453,  0.9998491 ,  0.28256902,
        -0.6076438 , -0.25311825,  0.43338263, -0.01120602, -0.5026296 ,
        -0.07193501,  0.33450103,  0.70257944, -0.1674666 , -0.14112465,
        -0.46272305, -0.10466173, -0.11190244,  0.9632924 , -0.352972  ,
        -0.09612415,  2.502883  ,  0.90979594,  0.06525096, -0.15650983,
         0.07649422,  0.5361454 , -0.15982322,  0.2726681 ,  0.7959145 ,
         1.0643854 , -0.13122362,  0.14212525,  1.7836213 ,  0.03983527,
         2.4179337 ,  0.9316936 , -0.37145653,  0.08994286,  1.3421928 ,
         0.06628615, -0.2548954 ,  0.73440653,  1.5245328 ,  2.0666215 ,
         0.8829979 ,  0.44813338, -0.09580511,  0.58701533,  0.45725712,
         3.1279764 ,  0.1818022 ,  0.69001496,  4.0144386 ,  2.7466989 ,
         0.6228162 ,  0.2926163 ,  0.73554444,  2.2512357 ,  0.70562387,
         1.9445065 ,  2.775793  ,  2.8211565 ,  0.27431905, -1.3267546 ,
        -1.6709023 , -2.2493708 , -3.419692  , -3.6

`build_embedding_dataframe`メソッドは埋め込み配列をデータフレームに変換して、次のLightGBMの学習で使いやすくする。

In [ ]:
def build_embedding_dataframe(
    race_ids: np.ndarray,
    race_embedding: np.ndarray,
    winner_scores: np.ndarray,
    winner_pred: np.ndarray,
    y: np.ndarray | None = None,
) -> pl.DataFrame:
    emb_cols = [f"tenkai_emb{i}" for i in range(race_embedding.shape[1])]
    score_cols = [f"winner_score_{i+1}" for i in range(CFG.num_boats)]
    payload: dict[str, Any] = {
        "RaceId": race_ids,
        **{col: race_embedding[:, i] for i, col in enumerate(emb_cols)},
        **{col: winner_scores[:, i] for i, col in enumerate(score_cols)},
        "winner_pred": winner_pred + 1,
    }
    if y is not None:
        payload["winner_true"] = y + 1
    return pl.DataFrame(payload).sort("RaceId")


emb_df = build_embedding_dataframe(
    race_ids=race_ids,
    race_embedding=race_embedding,
    winner_scores=winner_scores,
    winner_pred=winner_pred,
    y=y,
)
emb_df

RaceId,tenkai_emb0,tenkai_emb1,tenkai_emb2,tenkai_emb3,tenkai_emb4,tenkai_emb5,tenkai_emb6,tenkai_emb7,tenkai_emb8,tenkai_emb9,tenkai_emb10,tenkai_emb11,tenkai_emb12,tenkai_emb13,tenkai_emb14,tenkai_emb15,tenkai_emb16,tenkai_emb17,tenkai_emb18,tenkai_emb19,tenkai_emb20,tenkai_emb21,tenkai_emb22,tenkai_emb23,tenkai_emb24,tenkai_emb25,tenkai_emb26,tenkai_emb27,tenkai_emb28,tenkai_emb29,tenkai_emb30,tenkai_emb31,tenkai_emb32,tenkai_emb33,tenkai_emb34,tenkai_emb35,…,tenkai_emb90,tenkai_emb91,tenkai_emb92,tenkai_emb93,tenkai_emb94,tenkai_emb95,tenkai_emb96,tenkai_emb97,tenkai_emb98,tenkai_emb99,tenkai_emb100,tenkai_emb101,tenkai_emb102,tenkai_emb103,tenkai_emb104,tenkai_emb105,tenkai_emb106,tenkai_emb107,tenkai_emb108,tenkai_emb109,tenkai_emb110,tenkai_emb111,tenkai_emb112,tenkai_emb113,tenkai_emb114,tenkai_emb115,tenkai_emb116,tenkai_emb117,tenkai_emb118,winner_score_1,winner_score_2,winner_score_3,winner_score_4,winner_score_5,winner_score_6,winner_pred,winner_true
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64
"""20240314_03_01""",-0.426628,0.43723,-0.430415,0.999849,0.282569,-0.607644,-0.253118,0.433383,-0.011206,-0.50263,-0.071935,0.334501,0.702579,-0.167467,-0.141125,-0.462723,-0.104662,-0.111902,0.963292,-0.352972,-0.096124,2.502883,0.909796,0.065251,-0.15651,0.076494,0.536145,-0.159823,0.272668,0.795914,1.064385,-0.131224,0.142125,1.783621,0.039835,2.417934,…,0.180624,0.483607,0.335239,2.111767,0.323293,0.485911,1.295367,1.782956,0.307232,0.30749,0.465044,1.610859,0.732469,1.358966,1.888891,1.799854,0.322147,0.966768,0.375973,2.057696,0.613836,1.633702,0.352026,1.0,1.0,1.0,1.0,1.0,1.0,-1.326755,-2.249371,-1.670902,-3.419692,-3.838638,-3.691878,1,2
"""20240314_03_02""",-0.828758,0.291391,-0.391116,1.323153,0.796326,-0.791699,-0.27159,0.581845,-0.095451,-0.282981,-0.376962,-0.301457,1.420324,-0.001709,0.183267,-0.505175,-0.358191,-0.346908,1.365848,0.266842,-0.430074,3.662262,1.682066,-0.147153,-0.462275,0.026118,0.766586,-0.458218,-0.388045,0.984469,1.837832,-0.303645,-0.705177,1.59828,0.598001,2.546445,…,0.18583,0.2091,0.046652,1.571989,0.408007,0.095678,1.013291,1.377599,0.0439,0.096853,0.360586,1.077081,0.298205,0.843541,1.308402,1.310275,0.147607,1.494946,0.514975,0.854877,0.580299,1.3939,1.161003,1.0,1.0,1.0,1.0,1.0,1.0,-1.818383,-2.881825,-3.117704,-2.91681,-3.236833,-2.644773,1,1
"""20240314_03_03""",-0.759386,0.450601,-0.476064,1.287382,0.736362,-0.754438,-0.285746,0.544202,-0.090159,-0.417075,-0.306905,-0.021246,1.184298,0.072688,0.14355,-0.548355,-0.294234,-0.285129,1.180764,0.159786,-0.34861,3.322197,1.436715,-0.127397,-0.396632,0.059754,0.641193,-0.42346,-0.228243,0.89562,1.71256,-0.229486,-0.487068,1.994839,0.013078,2.638852,…,0.113043,0.26904,0.149086,1.770653,0.415436,0.223732,1.08117,1.511856,0.0763,0.163353,0.314438,1.381182,0.424163,1.010301,1.617015,1.589038,0.218543,1.392032,0.517275,1.514101,1.324773,0.950837,0.300982,1.0,1.0,1.0,1.0,1.0,1.0,-1.727282,-2.831368,-3.49676,-2.277541,-3.182346,-2.953429,1,5
"""20240314_03_04""",-0.727381,0.390365,-0.554309,1.326287,0.705528,-0.762093,-0.346559,0.614505,-0.077949,-0.385219,-0.422287,-0.076359,1.293152,0.190237,0.249618,-0.565901,-0.399677,-0.335665,1.084706,0.322003,-0.390962,3.708874,1.449554,-0.140962,-0.40746,0.090686,0.509774,-0.531336,-0.219113,1.079672,1.769586,-0.307674,-0.218561,1.272753,-0.382342,2.079124,…,0.085129,0.305915,0.06572,1.47308,0.374465,0.161232,1.153588,1.222526,0.054544,0.20295,0.535753,1.267548,0.512212,0.796261,1.378009,1.34366,0.247426,0.960716,1.626789,0.943533,0.972077,0.586591,0.910293,1.0,1.0,1.0,1.0,1.0,1.0,-2.703183,-1.993091,-2.982571,-2.687394,-2.812865,-3.655241,2,1
"""20240314_03_05""",-0.726947,0.235398,-0.532532,1.190563,0.684389,-0.705904,-0.328442,0.53547,-0.062205,-0.4760

予測時の埋め込みを得るときは、`load_model_checkpoint`メソッドで保存してある学習済みモデルを再現し、学習時に説明してた必要な処理を予測データに適用し、埋め込みを計算する。


In [135]:
def load_model_checkpoint(model) -> TenkaiGATEncoder:
    checkpoint = torch.load(CFG.model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    return model


model = load_model_checkpoint(model)

pred_df = data_processor.load_race_dataframe(CFG.pred_csv_path)
X_race, lane_race, _, _, race_ids, _ = data_processor.build_race_arrays(
    df=pred_df, with_label=False)  # fmt:skip

race_embedding, winner_scores, winner_pred = pred_embeddings(model, X_race, lane_race)
emb_df = build_embedding_dataframe(
    race_ids=race_ids,
    race_embedding=race_embedding,
    winner_scores=winner_scores,
    winner_pred=winner_pred,
    y=None,
)
emb_df

RaceId,tenkai_emb0,tenkai_emb1,tenkai_emb2,tenkai_emb3,tenkai_emb4,tenkai_emb5,tenkai_emb6,tenkai_emb7,tenkai_emb8,tenkai_emb9,tenkai_emb10,tenkai_emb11,tenkai_emb12,tenkai_emb13,tenkai_emb14,tenkai_emb15,tenkai_emb16,tenkai_emb17,tenkai_emb18,tenkai_emb19,tenkai_emb20,tenkai_emb21,tenkai_emb22,tenkai_emb23,tenkai_emb24,tenkai_emb25,tenkai_emb26,tenkai_emb27,tenkai_emb28,tenkai_emb29,tenkai_emb30,tenkai_emb31,tenkai_emb32,tenkai_emb33,tenkai_emb34,tenkai_emb35,…,tenkai_emb89,tenkai_emb90,tenkai_emb91,tenkai_emb92,tenkai_emb93,tenkai_emb94,tenkai_emb95,tenkai_emb96,tenkai_emb97,tenkai_emb98,tenkai_emb99,tenkai_emb100,tenkai_emb101,tenkai_emb102,tenkai_emb103,tenkai_emb104,tenkai_emb105,tenkai_emb106,tenkai_emb107,tenkai_emb108,tenkai_emb109,tenkai_emb110,tenkai_emb111,tenkai_emb112,tenkai_emb113,tenkai_emb114,tenkai_emb115,tenkai_emb116,tenkai_emb117,tenkai_emb118,winner_score_1,winner_score_2,winner_score_3,winner_score_4,winner_score_5,winner_score_6,winner_pred
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64
"""20260314_04_01""",-0.417532,1.122598,-0.418502,1.806692,-0.117156,-0.486375,-0.259593,0.813791,0.151798,-0.377751,-0.325495,0.666868,0.016792,0.344,0.092074,-0.494887,-0.246438,-0.176039,-0.549209,-0.011664,-0.163167,3.263174,-0.222164,0.015464,-0.17952,0.115305,-0.725547,-0.477691,0.640208,2.013496,0.053452,-0.192853,1.164756,1.806916,-0.03185,2.697544,…,0.572201,0.077723,0.543763,0.237604,0.696462,0.526117,0.391194,1.65197,0.593741,0.229238,0.45644,0.337874,0.54624,0.661417,0.898788,1.388508,0.768974,0.442681,1.260172,0.412954,0.894646,1.184981,1.010191,1.237055,1.0,1.0,1.0,1.0,1.0,1.0,-1.48516,-2.493447,-2.292649,-2.37887,-1.958839,-5.453536,1
"""20260314_04_02""",-0.738416,0.560594,-0.506543,1.497433,0.688249,-0.755399,-0.352424,0.587763,-0.09697,-0.374031,-0.373392,0.026348,1.13125,0.271547,0.178559,-0.572939,-0.358027,-0.346557,0.853466,0.362391,-0.452113,3.93709,1.406908,-0.145471,-0.391188,-0.10658,0.435253,-0.484972,-0.230976,1.178733,1.603179,-0.292557,-0.242033,1.403293,-0.224275,2.339366,…,0.262186,0.074603,0.331092,0.048347,1.675297,0.376689,0.060183,1.18502,1.162629,0.0478,0.182411,0.394716,1.421979,0.547338,0.783448,1.317106,1.211831,0.256852,0.832316,1.194542,0.956893,1.983431,0.512016,0.520802,1.0,1.0,1.0,1.0,1.0,1.0,-2.470511,-2.259804,-2.584037,-3.644814,-2.837371,-3.064296,2
"""20260314_04_03""",-0.607919,0.486321,-0.272641,1.087448,0.336426,-0.709149,-0.127199,0.498267,-0.037633,-0.548077,-0.167364,0.235886,0.836195,-0.007463,-0.089042,-0.33464,-0.193312,-0.176602,0.850679,-0.08822,-0.210673,2.879269,0.982273,0.00125,-0.256066,0.026301,0.435762,-0.304137,0.154205,0.944398,1.171148,-0.17266,0.188438,1.537854,1.016312,2.358133,…,0.422698,0.405384,0.450593,0.288585,2.037827,0.471451,0.386562,1.495441,1.713285,0.249802,0.264113,0.338175,1.609989,0.671171,1.312684,1.852138,1.747875,0.335155,1.4528,0.631609,0.689396,0.74247,0.941556,1.542169,1.0,1.0,1.0,1.0,1.0,1.0,-0.911417,-2.23155,-3.08178,-3.320891,-3.939262,-2.241939,1
"""20260314_04_04""",-0.655173,0.60752,-0.417039,1.458761,0.60775,-0.699637,-0.285557,0.674422,-0.050505,-0.42962,-0.358219,0.164321,1.072195,0.201058,0.198014,-0.503423,-0.344936,-0.332737,0.524987,0.270557,-0.414227,3.738466,1.019168,-0.131807,-0.349792,-0.239334,0.349446,-0.465489,0.040785,1.249582,1.467472,-0.277724,0.005846,1.384873,0.289288,2.627293,…,0.334135,0.124484,0.376594,0.028623,1.674105,0.431526,0.075675,1.342888,1.322441,0.028299,0.21513,0.18286,1.509922,0.593301,0.913488,1.529806,1.35304,0.286926,1.189857,0.86644,1.269103,0.768814,1.05127,0.854516,1.0,1.0,1.0,1.0,1.0,1.0,-2.001801,-2.448314,-2.456858,-3.219488,-2.652094,-3.796324,1
"""20260314_04_05""",-0.767857,0.398753,-0.42302,1.390794,0.749642,-0.767064,-0.266957,0.61